# FailureLLMUnlearning - Complete Setup and Execution Guide

This notebook provides a step-by-step guide to:
1. Clone the repository
2. Set up the conda environment
3. Install all dependencies
4. Load data from HuggingFace
5. Run unlearning methods
6. Evaluate the results

**Paper**: Catastrophic Failure of LLM Unlearning via Quantization (ICLR 2025)
**Repository**: https://github.com/zzwjames/FailureLLMUnlearning.git


## Step 1: Clone the Repository

First, we'll clone the repository if it doesn't already exist.


In [12]:
import shutil, pathlib

root = pathlib.Path("/workspace/CS534L_Project/FailureLLMUnlearning")
dirs_to_remove = [
    root / "data" / "news",
]

for d in dirs_to_remove:
    if d.exists():
        shutil.rmtree(d)
        print(f"Removed {d}")
    else:
        print(f"Not found: {d}")

Removed /workspace/CS534L_Project/FailureLLMUnlearning/data/news


In [2]:
import os
import subprocess
from pathlib import Path

# Set the project directory
project_dir = Path("/workspace/CS534L_Project")
repo_dir = project_dir / "FailureLLMUnlearning"
repo_url = "https://github.com/zzwjames/FailureLLMUnlearning.git"

# Check if repository already exists
if repo_dir.exists():
    print(f"Repository already exists at: {repo_dir}")
    print("Skipping clone step. If you want to re-clone, delete the directory first.")
else:
    print(f"Cloning repository from {repo_url}...")
    os.chdir(project_dir)
    result = subprocess.run(
        ["git", "clone", repo_url],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✓ Repository cloned successfully!")
    else:
        print(f"✗ Error cloning repository: {result.stderr}")
        raise Exception("Failed to clone repository")

# Change to repository directory
os.chdir(repo_dir)
print(f"\nCurrent working directory: {os.getcwd()}")


Repository already exists at: /workspace/CS534L_Project/FailureLLMUnlearning
Skipping clone step. If you want to re-clone, delete the directory first.

Current working directory: /workspace/CS534L_Project/FailureLLMUnlearning


## Step 2: Set Up Conda Environment

According to the README, we need to create a conda environment using the `environment.yml` file. This will create an environment named `py310` with Python 3.10 and all required dependencies.


In [3]:
# Check if conda is available
result = subprocess.run(["conda", "--version"], capture_output=True, text=True)
if result.returncode != 0:
    print("⚠ Warning: Conda is not available. Please install conda or use pip instead.")
    print("You can still proceed with pip installation in the next step.")
else:
    print(f"✓ Conda found: {result.stdout.strip()}")
    
# Check if environment.yml exists
env_file = repo_dir / "environment.yml"
if env_file.exists():
    print(f"✓ Found environment.yml at: {env_file}")
    print("\nTo create the conda environment, run the following commands in your terminal:")
    print(f"  conda env create -f {env_file}")
    print("  conda activate py310")
    print("\nOr if the environment already exists, just activate it:")
    print("  conda activate py310")
else:
    print("✗ environment.yml not found!")


FileNotFoundError: [Errno 2] No such file or directory: 'conda'

## Step 3: Install Dependencies

We'll install dependencies using pip. The repository provides both `environment.yml` (for conda) and `requirements.txt` (for pip). We'll use pip for compatibility.


In [4]:
# Read requirements.txt to see what will be installed
requirements_file = repo_dir / "requirements.txt"
if requirements_file.exists():
    print("Requirements from requirements.txt:")
    print("=" * 60)
    with open(requirements_file, 'r') as f:
        print(f.read())
    print("=" * 60)
    
    # Note: We'll install these in the next cell
    print("\n⚠ Note: Installing packages can take several minutes.")
    print("The main dependencies include:")
    print("  - torch==2.2")
    print("  - transformers==4.40")
    print("  - accelerate==0.29")
    print("  - datasets==2.19")
    print("  - bitsandbytes==0.42.0 (optional, requires CUDA/Linux)")
    print("  - and many more...")
else:
    print("✗ requirements.txt not found!")

Requirements from requirements.txt:
accelerate==0.29
bitsandbytes==0.42.0
datasets==2.19
einops==0.7
huggingface-hub>=0.26.0,<1.0
ipykernel==6.29
ipython==8.24
ipywidgets==8.1
matplotlib==3.9
matplotlib-inline==0.1
numpy==1.26
openai==1.23
pandas==2.2
peft==0.13.2
protobuf==5.26
python-dotenv==1.0
rouge==1.0.1
rouge-score==0.1
scienceplots==2.1
scikit-learn==1.4
scipy==1.13
seaborn==0.13
sympy==1.12
tokenizers==0.19
torch==2.2
tqdm
transformers==4.40
trl>=0.8.1


⚠ Note: Installing packages can take several minutes.
The main dependencies include:
  - torch==2.2
  - transformers==4.40
  - accelerate==0.29
  - datasets==2.19
  - bitsandbytes==0.42.0 (optional, requires CUDA/Linux)
  - and many more...


In [5]:
# Install dependencies
# This may take 10-20 minutes depending on your internet connection

import platform
import sys

print("Installing dependencies from requirements.txt...")
print("This may take several minutes. Please be patient...\n")

# Check if we're on macOS (bitsandbytes doesn't work on macOS/ARM)
is_macos = platform.system() == "Darwin"
is_arm = platform.machine() == "arm64"

if is_macos:
    print("⚠ Detected macOS system.")
    print("Note: bitsandbytes requires CUDA (Linux/NVIDIA GPU) and won't work on macOS.")
    print("The code will work in full-precision mode without bitsandbytes.\n")

# Create a temporary requirements file without bitsandbytes for initial installation
temp_requirements = repo_dir / "requirements_temp.txt"
with open(requirements_file, 'r') as f:
    lines = f.readlines()

# Filter out bitsandbytes line
filtered_lines = [line for line in lines if not line.strip().startswith('bitsandbytes')]

with open(temp_requirements, 'w') as f:
    f.writelines(filtered_lines)

print("Step 1: Installing core dependencies (excluding bitsandbytes)...")
result = subprocess.run(
    ["pip", "install", "-r", str(temp_requirements)],
    cwd=str(repo_dir),
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✓ Core dependencies installed successfully!")
else:
    print("✗ Error installing core dependencies:")
    print(result.stderr)
    print("\nTrying to continue anyway...")

# Try to install bitsandbytes separately (will fail on macOS, which is OK)
print("\nStep 2: Attempting to install bitsandbytes (optional for quantization)...")
bitsandbytes_result = subprocess.run(
    ["pip", "install", "bitsandbytes==0.42.0"],
    cwd=str(repo_dir),
    capture_output=True,
    text=True
)

if bitsandbytes_result.returncode == 0:
    print("✓ bitsandbytes installed successfully!")
    print("  You can use 4-bit and 8-bit quantization.")
elif is_macos:
    print("⚠ bitsandbytes installation skipped (not available on macOS).")
    print("  This is expected. You can still run the code in full-precision mode.")
    print("  Set quantize_4bit=0 and quantize_8bit=0 in evaluation.")
else:
    print("⚠ bitsandbytes installation failed:")
    print(bitsandbytes_result.stderr)
    print("  You can still run the code in full-precision mode.")

# Clean up temp file
if temp_requirements.exists():
    temp_requirements.unlink()

print("\n" + "="*60)
print("Installation summary:")
print("  - Core dependencies: ✓")
if bitsandbytes_result.returncode == 0:
    print("  - bitsandbytes: ✓ (quantization available)")
else:
    print("  - bitsandbytes: ✗ (quantization not available, use full-precision)")
print("="*60)

Installing dependencies from requirements.txt...
This may take several minutes. Please be patient...

Step 1: Installing core dependencies (excluding bitsandbytes)...
✓ Core dependencies installed successfully!

Step 2: Attempting to install bitsandbytes (optional for quantization)...
✓ bitsandbytes installed successfully!
  You can use 4-bit and 8-bit quantization.

Installation summary:
  - Core dependencies: ✓
  - bitsandbytes: ✓ (quantization available)


In [6]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device in use:", device)
if device.type == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))

    print("GPU memory (MB):", torch.cuda.get_device_properties(0).total_memory // (1024**2))

Device in use: cuda
GPU name: NVIDIA A100-SXM4-80GB
GPU memory (MB): 81153


## Step 4: Load Data from HuggingFace

According to the README, we need to load data from HuggingFace datasets. This will download:
- MUSE-News dataset and target model
- MUSE-Books dataset and target model

The data will be saved in the `data/` directory.


## Step 4a: Authenticate with HuggingFace

Some HuggingFace datasets require authentication. Let's check if you're logged in and authenticate if needed.


In [ ]:
# Check HuggingFace authentication and login if needed
import subprocess
import os

print("Checking HuggingFace authentication...\n")

# Check if huggingface_hub is installed
try:
    from huggingface_hub import whoami, login
    hf_available = True
except ImportError:
    print("⚠ huggingface_hub not found. Installing...")
    subprocess.run(["pip", "install", "huggingface_hub"], capture_output=True)
    from huggingface_hub import whoami, login
    hf_available = True

# Try to get current user info
try:
    user_info = whoami()
    print(f"✓ Already authenticated as: {user_info.get('name', 'Unknown')}")
    print("  You can proceed to load data.\n")
    authenticated = True
except Exception as e:
    print("⚠ Not authenticated with HuggingFace.")
    print("\nTo authenticate, you have two options:\n")
    print("Option 1: Login via CLI (Recommended)")
    print("  Run this command in your terminal:")
    print("    huggingface-cli login")
    print("  Then paste your access token when prompted.\n")
    print("Option 2: Login programmatically")
    print("  Uncomment the code below and provide your token:\n")
    print("  # token = 'your_huggingface_token_here'")
    print("  # login(token=token)")
    print("\nTo get your token:")
    print("  1. Go to https://huggingface.co/settings/tokens")
    print("  2. Create a new token (or use an existing one)")
    print("  3. Copy the token and use it above\n")
    
    # Uncomment and set your token here if you want to login programmatically
    authenticated = False
    token = "token read"
    login(token=token)
    print("✓ Authenticated successfully!")
    authenticated = True

Checking HuggingFace authentication...

✓ Already authenticated as: himishra
  You can proceed to load data.



In [33]:
# Check if data already exists
data_dir = repo_dir / "data"
if data_dir.exists() and any(data_dir.iterdir()):
    print("✓ Data directory already exists with content.")
    print("Skipping data download. If you want to re-download, delete the data/ directory first.")
    print(f"\nData directory contents:")
    for item in sorted(data_dir.iterdir()):
        if item.is_dir():
            print(f"  📁 {item.name}/")
else:
    print("Data directory is empty or doesn't exist.")
    print("Will load data from HuggingFace in the next cell.")

✓ Data directory already exists with content.
Skipping data download. If you want to re-download, delete the data/ directory first.

Data directory contents:
  📁 books/
  📁 news/


In [9]:
import sys
!{sys.executable} -m pip install hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 21.5 MB/s  0:00:00 eta 0:00:01


In [35]:
# Load data from HuggingFace
# This will download datasets for both News and Books corpora
# This may take some time depending on your internet connection

print("Loading data from HuggingFace...")
print("This will download:")
print("  - MUSE-News dataset (knowmem, verbmem, privleak, raw, scal, sust)")
print("  - MUSE-Books dataset (knowmem, verbmem, privleak, raw)")
print("\nThis may take 5-15 minutes depending on your connection...\n")

result = subprocess.run(
    ["python", "load_data.py"],
    cwd=str(repo_dir),
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✓ Data loaded successfully!")
    print("\nData structure:")
    print(result.stdout)
else:
    print("✗ Error loading data:")
    print(result.stderr)
    print("\nYou may need to:")
    print("  1. Check your internet connection")
    print("  2. Ensure you have HuggingFace access")
    print("  3. Install huggingface-hub: pip install huggingface-hub")


Loading data from HuggingFace...
This will download:
  - MUSE-News dataset (knowmem, verbmem, privleak, raw, scal, sust)
  - MUSE-Books dataset (knowmem, verbmem, privleak, raw)

This may take 5-15 minutes depending on your connection...

✓ Data loaded successfully!

Data structure:



## Step 4b: Load Twitter Misinformation Dataset from Hugging Face

This section loads the Twitter misinformation dataset from Hugging Face (`kanwal-mehreen18/twitter-misinformation-unlearning-eval`) and converts the parquet files to a format usable for finetuning.

The dataset contains three parquet files:
- `forget`: Data to be forgotten
- `retain1`: First retain set
- `retain2`: Second retain set

Run the next cell to download and convert these files.


In [8]:
# Load Twitter misinformation dataset from Hugging Face

from datasets import load_dataset
from utils import write_json, write_text
import os

# Configuration
HF_REPO_ID = "kanwal-mehreen18/twitter-misinformation-unlearning-eval"
OUTPUT_DIR = repo_dir / "data" / "twitter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# File names in the dataset (these become split names)
files = ["forget", "retain1", "retain2"]

print(f"Loading Twitter dataset from: {HF_REPO_ID}")
print(f"Output directory: {OUTPUT_DIR}\n")

try:
    # Load the entire dataset - the three parquet files become separate splits
    # Note: The parquet files are at the root level, not in a subdirectory
    print("Loading dataset (this may take a moment)...")
    full_dataset = load_dataset(HF_REPO_ID)
    
    print(f"Dataset loaded! Available splits: {list(full_dataset.keys())}\n")
    
    # Process each file/split
    for file_name in files:
        print(f"Processing {file_name}...")
        
        if file_name not in full_dataset:
            print(f"  ⚠ Warning: Split '{file_name}' not found in dataset")
            print(f"  Available splits: {list(full_dataset.keys())}")
            continue
        
        dataset = full_dataset[file_name]
        
        # Check what columns are available
        if len(dataset) > 0:
            print(f"  Columns: {dataset.column_names}")
            print(f"  Number of samples: {len(dataset)}")
            
            # Try to find the text column
            text_col = None
            for col in ['text', 'content', 'tweet', 'message', 'input', 'prompt']:
                if col in dataset.column_names:
                    text_col = col
                    break
            
            if text_col is None:
                # Use the first string column
                for col in dataset.column_names:
                    if len(dataset) > 0 and isinstance(dataset[0][col], str):
                        text_col = col
                        break
            
            if text_col is None:
                print(f"  ⚠ Warning: Could not find text column. Available: {dataset.column_names}")
                if len(dataset.column_names) > 0:
                    text_col = dataset.column_names[0]
                    print(f"  Using first column: {text_col}")
                else:
                    print(f"  ✗ No columns found, skipping {file_name}")
                    continue
            
            print(f"  Using column '{text_col}' as text source")
            
            # Extract text data (filter out None values)
            texts = []
            for item in dataset:
                text_val = item.get(text_col)
                if text_val is not None and isinstance(text_val, str) and len(text_val.strip()) > 0:
                    texts.append(text_val)
            
            if len(texts) == 0:
                print(f"  ⚠ Warning: No valid text found in {file_name}")
                continue
            
            # Save as JSON (list of strings)
            write_json(texts, str(OUTPUT_DIR / f"{file_name}.json"))
            
            # Save as TXT (concatenated with double newlines)
            write_text("\n\n".join(texts), str(OUTPUT_DIR / f"{file_name}.txt"))
            
            print(f"  ✓ Saved {len(texts)} samples to:")
            print(f"    - {OUTPUT_DIR}/{file_name}.json")
            print(f"    - {OUTPUT_DIR}/{file_name}.txt\n")
        else:
            print(f"  ⚠ Warning: {file_name} is empty\n")
    
    print("✓ Twitter dataset loading completed!")
    print(f"\nFiles available at: {OUTPUT_DIR}")
    print("  - forget.txt / forget.json")
    print("  - retain1.txt / retain1.json")
    print("  - retain2.txt / retain2.json")
            
except Exception as e:
    print(f"✗ Error loading dataset: {e}")
    import traceback
    traceback.print_exc()
    print("\nTroubleshooting:")
    print("  1. Make sure you're logged in: huggingface-cli login")
    print("  2. Check if the dataset is public or you have access")
    print("  3. Try inspecting the dataset structure:")
    print(f"     from datasets import load_dataset")
    print(f"     ds = load_dataset('{HF_REPO_ID}')")
    print(f"     print(ds)")
    print(f"     print(list(ds.keys()))")


Loading Twitter dataset from: kanwal-mehreen18/twitter-misinformation-unlearning-eval
Output directory: /workspace/CS534L_Project/FailureLLMUnlearning/data/twitter

Loading dataset (this may take a moment)...
Dataset loaded! Available splits: ['forget', 'retain1', 'retain2']

Processing forget...
  Columns: ['text', 'label']
  Number of samples: 1693
  Using column 'text' as text source
  ✓ Saved 1668 samples to:
    - /workspace/CS534L_Project/FailureLLMUnlearning/data/twitter/forget.json
    - /workspace/CS534L_Project/FailureLLMUnlearning/data/twitter/forget.txt

Processing retain1...
  Columns: ['text', 'label']
  Number of samples: 3386
  Using column 'text' as text source
  ✓ Saved 3386 samples to:
    - /workspace/CS534L_Project/FailureLLMUnlearning/data/twitter/retain1.json
    - /workspace/CS534L_Project/FailureLLMUnlearning/data/twitter/retain1.txt

Processing retain2...
  Columns: ['text', 'label']
  Number of samples: 3386
  Using column 'text' as text source
  ✓ Saved 3386

## Step 5c: Finetune Llama Model on Twitter Dataset

This section shows how to finetune a Llama model on the Twitter misinformation dataset using the converted files.

You can finetune on:
- **Forget set**: To create a reinforced model (for Task Vector method)
- **Retain set**: To preserve utility
- **Combined**: Merge forget + retain for general finetuning

Run the configuration cell first, then the finetuning cell.


In [9]:
# Configuration for finetuning on Twitter dataset

import os
from pathlib import Path

# Choose which dataset to finetune on
# Options: 'forget', 'retain1', 'retain2', or 'combined'
FINETUNE_DATASET = 'combined'  # Change this as needed

# Base model to finetune
BASE_MODEL_DIR = 'meta-llama/Llama-2-7b-hf'  # or path to your base model
TOKENIZER_DIR_FT = 'meta-llama/Llama-2-7b-hf'

# Output directory for finetuned model
if FINETUNE_DATASET == 'combined':
    # Combine forget + retain1 for combined training
    FINETUNE_OUT_DIR = repo_dir / "ckpt" / "twitter" / "finetuned_combined"
    FINETUNE_DATA_FILE = None  # Will be created from combined data
else:
    FINETUNE_OUT_DIR = repo_dir / "ckpt" / "twitter" / f"finetuned_{FINETUNE_DATASET}"
    FINETUNE_DATA_FILE = repo_dir / "data" / "twitter" / f"{FINETUNE_DATASET}.txt"

# Finetuning hyperparameters
FT_EPOCHS = 5
FT_BATCH_SIZE = 3
FT_LEARNING_RATE = "1e-5"
FT_MAX_LEN = 2048

# GPU configuration
CUDA_VISIBLE_DEVICES_FT = "0"
if CUDA_VISIBLE_DEVICES_FT:
    os.environ['CUDA_VISIBLE_DEVICES'] = CUDA_VISIBLE_DEVICES_FT

print("Finetuning Configuration:")
print("=" * 60)
print(f"Dataset: {FINETUNE_DATASET}")
print(f"Base Model: {BASE_MODEL_DIR}")
print(f"Output Directory: {FINETUNE_OUT_DIR}")
print(f"Data File: {FINETUNE_DATA_FILE}")
print(f"Epochs: {FT_EPOCHS}")
print(f"Batch Size: {FT_BATCH_SIZE}")
print(f"Learning Rate: {FT_LEARNING_RATE}")
print(f"Max Length: {FT_MAX_LEN}")
print("=" * 60)

# If combined, create a combined file
if FINETUNE_DATASET == 'combined':
    from utils import read_text, write_text
    forget_text = read_text(str(repo_dir / "data" / "twitter" / "forget.txt"))
    retain1_text = read_text(str(repo_dir / "data" / "twitter" / "retain1.txt"))
    combined_text = forget_text + "\n\n" + retain1_text
    FINETUNE_DATA_FILE = repo_dir / "data" / "twitter" / "combined.txt"
    write_text(combined_text, str(FINETUNE_DATA_FILE))
    print(f"\n✓ Created combined dataset: {FINETUNE_DATA_FILE}")


Finetuning Configuration:
Dataset: combined
Base Model: meta-llama/Llama-2-7b-hf
Output Directory: /workspace/CS534L_Project/FailureLLMUnlearning/ckpt/twitter/finetuned_combined
Data File: None
Epochs: 5
Batch Size: 3
Learning Rate: 1e-5
Max Length: 2048

✓ Created combined dataset: /workspace/CS534L_Project/FailureLLMUnlearning/data/twitter/combined.txt


In [10]:
# Run finetuning on Twitter dataset

import subprocess

# Import finetune function from baselines
import sys
sys.path.insert(0, str(repo_dir / "baselines"))

from baselines import finetune

print("Starting finetuning...")
print(f"Model: {BASE_MODEL_DIR}")
print(f"Data: {FINETUNE_DATA_FILE}")
print(f"Output: {FINETUNE_OUT_DIR}\n")

try:
    finetune(
        model_dir=BASE_MODEL_DIR,
        data_file=str(FINETUNE_DATA_FILE),
        out_dir=str(FINETUNE_OUT_DIR),
        epochs=FT_EPOCHS,
        per_device_batch_size=FT_BATCH_SIZE,
        learning_rate=float(FT_LEARNING_RATE),
        max_len=FT_MAX_LEN,
        tokenizer_dir=TOKENIZER_DIR_FT
    )
    
    print(f"\n✓ Finetuning completed successfully!")
    print(f"Finetuned model saved to: {FINETUNE_OUT_DIR}")
    print("\nYou can now use this model for:")
    print("  - Task Vector unlearning (if finetuned on forget set)")
    print("  - Further training or evaluation")
    
except Exception as e:
    print(f"✗ Error during finetuning: {e}")
    import traceback
    traceback.print_exc()

Starting finetuning...
Model: meta-llama/Llama-2-7b-hf
Data: /workspace/CS534L_Project/FailureLLMUnlearning/data/twitter/combined.txt
Output: /workspace/CS534L_Project/FailureLLMUnlearning/ckpt/twitter/finetuned_combined

🧹 Cleared GPU cache


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✓ Gradient checkpointing enabled (saves memory)

📊 GPU Memory: 6.31 GB / 79.25 GB allocated

🚀 Starting finetuning for 5 epochs...
   Dataset size: 708 samples
   Batch size per device: 3
   Gradient accumulation steps: 1
   Effective batch size: 3
   Model will be saved only at the end (no checkpoints)

💡 If you get CUDA OOM errors, try:
   - Reducing batch size (currently: 3)
   - Increasing gradient_accumulation_steps (currently: 1)
   - Reducing max_len (currently: 2048)


Starting Epoch 1/5


Step,Training Loss
1,2.029900
10,2.014700
20,1.912800
30,2.000600
40,1.950500
50,1.900900
60,1.940200
70,1.877900
80,1.856600
90,1.883500



✓ Completed Epoch 1/5
  📉 Epoch Loss: 1.912800


Starting Epoch 2/5

✓ Completed Epoch 2/5
  📉 Epoch Loss: 1.825000


Starting Epoch 3/5

✓ Completed Epoch 3/5
  📉 Epoch Loss: 1.721000


Starting Epoch 4/5

✓ Completed Epoch 4/5
  📉 Epoch Loss: 1.707200


Starting Epoch 5/5

✓ Completed Epoch 5/5
  📉 Epoch Loss: 1.648700


💾 Saving final model to /workspace/CS534L_Project/FailureLLMUnlearning/ckpt/twitter/finetuned_combined...

📊 Training Loss Summary:
   Epoch 1: Loss = 2.029900
   Epoch 1: Loss = 2.014700
   Epoch 1: Loss = 1.912800
   Epoch 1: Loss = 2.000600
   Epoch 1: Loss = 1.950500
   Epoch 1: Loss = 1.900900
   Epoch 1: Loss = 1.940200
   Epoch 1: Loss = 1.877900
   Epoch 1: Loss = 1.856600
   Epoch 1: Loss = 1.883500
   Epoch 1: Loss = 1.870700
   Epoch 1: Loss = 1.883800
   Epoch 1: Loss = 1.949300
   Epoch 1: Loss = 1.825900
   Epoch 1: Loss = 1.868800
   Epoch 1: Loss = 1.825300
   Epoch 1: Loss = 1.838200
   Epoch 1: Loss = 1.840800
   Epoch 1: Loss = 1.867500
   Epoch 1

## Step 5: Run Unlearning Methods

Now we'll run the unlearning methods. According to the README, we can use various algorithms:
- `ga`: Gradient Ascent
- `ga_gdr`: GA with Gradient Difference Regularization
- `ga_klr`: GA with KL Regularization
- `npo`: Negative Preference Optimization
- `npo_gdr`: NPO with GDR
- `npo_klr`: NPO with KLR
- `ga_gdr_sure`, `ga_klr_sure`, `npo_gdr_sure`, `npo_klr_sure`: SURE variants
- `rmu`: Retraining with Modified Updates

We'll start with a simple example using the `ga` algorithm on the News corpus.


In [9]:
# Configuration for unlearning
import os

# Set corpus (options: 'news' or 'books')
CORPUS = 'news'  # Change to 'books' if you want to use Books corpus

# Set algorithm (options: 'ga', 'ga_gdr', 'ga_klr', 'npo', 'npo_gdr', 'npo_klr', 
#                        'ga_gdr_sure', 'ga_klr_sure', 'npo_gdr_sure', 'npo_klr_sure', 'rmu')
ALGO = 'ga_gdr'  # Start with simple gradient ascent

# Paths
FORGET = f"data/{CORPUS}/raw/forget.txt"
RETAIN = f"data/{CORPUS}/raw/retain1.txt"
TARGET_DIR = f'muse-bench/MUSE-{CORPUS.capitalize()}_target'
TOKENIZER_DIR = 'meta-llama/Llama-2-7b-hf'
OUT_DIR = f"./ckpt/{CORPUS}/{ALGO}"

# Hyperparameters
MAX_LEN = 2048
EPOCHS = 10  # You may want to reduce this for testing (e.g., 1-2 epochs)
LR = '1e-5'
PER_DEVICE_BATCH_SIZE = 2
ALPHA = 1  # Weight for utility constraint
THRESHOLD = 90  # Threshold to filter out salient modules

# GPU configuration (set to empty string if no GPU available)
# For multi-GPU, use: "0,1,2,3"
CUDA_VISIBLE_DEVICES = "0"  # Adjust based on your GPU availability

print("Unlearning Configuration:")
print("=" * 60)
print(f"Corpus: {CORPUS}")
print(f"Algorithm: {ALGO}")
print(f"Target Model: {TARGET_DIR}")
print(f"Forget Data: {FORGET}")
print(f"Retain Data: {RETAIN}")
print(f"Output Directory: {OUT_DIR}")
print(f"Epochs: {EPOCHS}")
print(f"Learning Rate: {LR}")
print(f"Batch Size: {PER_DEVICE_BATCH_SIZE}")
print(f"GPU Devices: {CUDA_VISIBLE_DEVICES if CUDA_VISIBLE_DEVICES else 'CPU'}")
print("=" * 60)

Unlearning Configuration:
Corpus: news
Algorithm: ga_gdr
Target Model: muse-bench/MUSE-News_target
Forget Data: data/news/raw/forget.txt
Retain Data: data/news/raw/retain1.txt
Output Directory: ./ckpt/news/ga_gdr
Epochs: 10
Learning Rate: 1e-5
Batch Size: 2
GPU Devices: 0


In [12]:
print(f"PER_DEVICE_BATCH_SIZE: {PER_DEVICE_BATCH_SIZE}")
print(f"MAX_LEN: {MAX_LEN}")
print(f"EPOCHS: {EPOCHS}")
print(f"ALGO: {ALGO}")

PER_DEVICE_BATCH_SIZE: 2
MAX_LEN: 2048
EPOCHS: 10
ALGO: ga_gdr


In [13]:
print(f"TARGET_DIR: {TARGET_DIR}")
print(f"TOKENIZER_DIR: {TOKENIZER_DIR}")

TARGET_DIR: muse-bench/MUSE-News_target
TOKENIZER_DIR: meta-llama/Llama-2-7b-hf


In [11]:
!nvidia-smi

Mon Dec  1 09:46:27 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.57.01              Driver Version: 565.57.01      CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:0B:00.0 Off |                    0 |
| N/A   25C    P0             64W /  400W |       4MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [10]:
# Run unlearning
# Note: This will take a significant amount of time (potentially hours depending on epochs and GPU)
# Make sure you have sufficient GPU memory and time

import subprocess

# Set environment variable for GPU
if CUDA_VISIBLE_DEVICES:
    os.environ['CUDA_VISIBLE_DEVICES'] = CUDA_VISIBLE_DEVICES

# Build the command
unlearn_script = repo_dir / "baselines" / "unlearn.py"

cmd = [
    "python", str(unlearn_script),
    "--algo", ALGO,
    "--model_dir", TARGET_DIR,
    "--tokenizer_dir", TOKENIZER_DIR,
    "--data_file", FORGET,
    "--retain_data_file", RETAIN,
    "--out_dir", OUT_DIR,
    "--max_len", str(MAX_LEN),
    "--epochs", str(EPOCHS),
    "--lr", LR,
    "--alpha", str(ALPHA),
    "--threshold", str(THRESHOLD),
    "--per_device_batch_size", str(PER_DEVICE_BATCH_SIZE)
]

print("Running unlearning...")
print(f"Command: {' '.join(cmd)}\n")
print("⚠ This may take a long time (hours for full training).")
print("⚠ Make sure you have:")
print("   - Sufficient GPU memory (recommended: 16GB+ VRAM)")
print("   - HuggingFace access to download models")
print("   - Stable internet connection\n")

# Uncomment the following lines to actually run the unlearning
result = subprocess.run(
    cmd,
    cwd=str(repo_dir),
    capture_output=False,  # Set to True if you want to capture output
    text=True
)

if result.returncode == 0:
    print("✓ Unlearning completed successfully!")
    print(f"Model saved to: {OUT_DIR}")
else:
    print("✗ Error during unlearning:")
    print(result.stderr)

print("\n⚠ To run unlearning, uncomment the code above and execute this cell.")
print("For now, we'll proceed with evaluation assuming you have a trained model.")


Running unlearning...
Command: python /workspace/CS534L_Project/FailureLLMUnlearning/baselines/unlearn.py --algo ga_gdr --model_dir muse-bench/MUSE-News_target --tokenizer_dir meta-llama/Llama-2-7b-hf --data_file data/news/raw/forget.txt --retain_data_file data/news/raw/retain1.txt --out_dir ./ckpt/news/ga_gdr --max_len 2048 --epochs 10 --lr 1e-5 --alpha 1 --threshold 90 --per_device_batch_size 2

⚠ This may take a long time (hours for full training).
⚠ Make sure you have:
   - Sufficient GPU memory (recommended: 16GB+ VRAM)
   - HuggingFace access to download models
   - Stable internet connection



/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
  0%|          | 1/2040 [00:02<1:19:37,  2.34s/it]Traceback (most recent call last):
  File "/workspace/CS534L_Project/FailureLLMUnlearning/baselines/unlearn.py", line 118, in <module>
    main()
  File "/workspace/CS534L_Project/FailureLLMUnlearning/baselines/unlearn.py", line 38, in main
    it_unlearn(
  File "/workspace/CS534L_Project/FailureLLMUnlearning/baselines/baselines/iterative.py", line 89, in unlearn
    trainer.train(resume_from_checkpoint=resume_from_checkpoint)
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 1859, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", lin

✗ Error during unlearning:
None

⚠ To run unlearning, uncomment the code above and execute this cell.
For now, we'll proceed with evaluation assuming you have a trained model.


## Step 5b: Run GA + GDR + Hinge Loss (ga_gdr_q4)

This section demonstrates the new **GA + GDR + Hinge Loss** algorithm (`ga_gdr_q4`), which combines:
- **GA (Gradient Ascent)**: Maximizes loss on forget set
- **GDR (Gradient Difference Regularization)**: Preserves utility on retain set
- **Hinge Loss (q4)**: Quantization-aware hinge loss that enforces logit separation between unlearned and reference model

The hinge loss term encourages the model to have logit differences of at least `delta_q/2` from the reference model, which helps with quantization robustness.


In [36]:
# Configuration for GA + GDR + Hinge Loss (ga_gdr_q4) unlearning
import os

# Set corpus (options: 'news' or 'books')
CORPUS_Q4 = 'news'  # Change to 'books' if you want to use Books corpus

# Set algorithm to ga_gdr_q4 (GA + GDR + Hinge Loss)
ALGO_Q4 = 'ga_gdr_q4'

# Paths
FORGET_Q4 = f"data/{CORPUS_Q4}/raw/forget.txt"
RETAIN_Q4 = f"data/{CORPUS_Q4}/raw/retain1.txt"
TARGET_DIR_Q4 = f'muse-bench/MUSE-{CORPUS_Q4.capitalize()}_target'
TOKENIZER_DIR_Q4 = 'meta-llama/Llama-2-7b-hf'
OUT_DIR_Q4 = f"./ckpt/{CORPUS_Q4}/{ALGO_Q4}"

# Hyperparameters
MAX_LEN_Q4 = 2048
EPOCHS_Q4 = 10  # You may want to reduce this for testing (e.g., 1-2 epochs)
LR_Q4 = '1e-5'
PER_DEVICE_BATCH_SIZE_Q4 = 2
ALPHA_Q4 = 5  # Weight for utility constraint (GDR term)
THRESHOLD_Q4 = 90  # Threshold to filter out salient modules (not used for non-SURE)

# Hinge Loss hyperparameters (NEW)
LAMBDA_Q = 1.0  # Weight for quantization-aware hinge loss on forget set
DELTA_Q = 1.0   # Logit-space bucket width Delta (margin = Delta/2)

# GPU configuration
CUDA_VISIBLE_DEVICES_Q4 = "0"  # Adjust based on your GPU availability

print("GA + GDR + Hinge Loss (ga_gdr_q4) Configuration:")
print("=" * 60)
print(f"Corpus: {CORPUS_Q4}")
print(f"Algorithm: {ALGO_Q4}")
print(f"Target Model: {TARGET_DIR_Q4}")
print(f"Forget Data: {FORGET_Q4}")
print(f"Retain Data: {RETAIN_Q4}")
print(f"Output Directory: {OUT_DIR_Q4}")
print(f"Epochs: {EPOCHS_Q4}")
print(f"Learning Rate: {LR_Q4}")
print(f"Batch Size: {PER_DEVICE_BATCH_SIZE_Q4}")
print(f"Alpha (GDR weight): {ALPHA_Q4}")
print(f"Lambda_q (Hinge weight): {LAMBDA_Q}")
print(f"Delta_q (Hinge margin): {DELTA_Q}")
print(f"GPU Devices: {CUDA_VISIBLE_DEVICES_Q4 if CUDA_VISIBLE_DEVICES_Q4 else 'CPU'}")
print("=" * 60)


GA + GDR + Hinge Loss (ga_gdr_q4) Configuration:
Corpus: news
Algorithm: ga_gdr_q4
Target Model: muse-bench/MUSE-News_target
Forget Data: data/news/raw/forget.txt
Retain Data: data/news/raw/retain1.txt
Output Directory: ./ckpt/news/ga_gdr_q4
Epochs: 10
Learning Rate: 1e-5
Batch Size: 2
Alpha (GDR weight): 5
Lambda_q (Hinge weight): 1.0
Delta_q (Hinge margin): 1.0
GPU Devices: 0


In [ ]:
# Run GA + GDR + Hinge Loss (ga_gdr_q4) unlearning
# Note: This will take a significant amount of time (potentially hours depending on epochs and GPU)
# Make sure you have sufficient GPU memory and time

import subprocess

# Set environment variable for GPU
if CUDA_VISIBLE_DEVICES_Q4:
    os.environ['CUDA_VISIBLE_DEVICES'] = CUDA_VISIBLE_DEVICES_Q4

# Build the command with new hinge loss parameters
unlearn_script = repo_dir / "baselines" / "unlearn.py"

cmd_q4 = [
    "python", str(unlearn_script),
    "--algo", ALGO_Q4,
    "--model_dir", TARGET_DIR_Q4,
    "--tokenizer_dir", TOKENIZER_DIR_Q4,
    "--data_file", FORGET_Q4,
    "--retain_data_file", RETAIN_Q4,
    "--out_dir", OUT_DIR_Q4,
    "--max_len", str(MAX_LEN_Q4),
    "--epochs", str(EPOCHS_Q4),
    "--lr", LR_Q4,
    "--alpha", str(ALPHA_Q4),
    "--threshold", str(THRESHOLD_Q4),
    "--per_device_batch_size", str(PER_DEVICE_BATCH_SIZE_Q4),
    "--lambda_q", str(LAMBDA_Q),  # NEW: Hinge loss weight
    "--delta_q", str(DELTA_Q)     # NEW: Hinge loss margin
]

print("Running GA + GDR + Hinge Loss unlearning...")
print(f"Command: {' '.join(cmd_q4)}\n")
print("⚠ This may take a long time (hours for full training).")
print("⚠ Make sure you have:")
print("   - Sufficient GPU memory (recommended: 16GB+ VRAM)")
print("   - HuggingFace access to download models")
print("   - Stable internet connection")
print(f"\n📝 Algorithm details:")
print(f"   - GA: Maximizes loss on forget set")
print(f"   - GDR: Preserves utility on retain set (weight: {ALPHA_Q4})")
print(f"   - Hinge Loss: Enforces logit separation (weight: {LAMBDA_Q}, margin: {DELTA_Q/2})\n")

# Run the unlearning
result_q4 = subprocess.run(
    cmd_q4,
    cwd=str(repo_dir),
    capture_output=False,  # Set to True if you want to capture output
    text=True
)

if result_q4.returncode == 0:
    print("✓ GA + GDR + Hinge Loss unlearning completed successfully!")
    print(f"Model saved to: {OUT_DIR_Q4}")
else:
    print("✗ Error during unlearning:")
    print(result_q4.stderr)


Running GA + GDR + Hinge Loss unlearning...
Command: python /workspace/CS534L_Project/FailureLLMUnlearning/baselines/unlearn.py --algo ga_gdr_q4 --model_dir muse-bench/MUSE-News_target --tokenizer_dir meta-llama/Llama-2-7b-hf --data_file data/news/raw/forget.txt --retain_data_file data/news/raw/retain1.txt --out_dir ./ckpt/news/ga_gdr_q4 --max_len 2048 --epochs 10 --lr 1e-5 --alpha 5 --threshold 90 --per_device_batch_size 2 --lambda_q 1.0 --delta_q 1.0

⚠ This may take a long time (hours for full training).
⚠ Make sure you have:
   - Sufficient GPU memory (recommended: 16GB+ VRAM)
   - HuggingFace access to download models
   - Stable internet connection

📝 Algorithm details:
   - GA: Maximizes loss on forget set
   - GDR: Preserves utility on retain set (weight: 5)
   - Hinge Loss: Enforces logit separation (weight: 1.0, margin: 0.5)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

## Step 6: Evaluate Unlearned Models

After training, we need to evaluate the unlearned models using various metrics:
- `verbmem_f`: VerbMem Forget (measures if forgotten content is still generated)
- `privleak`: PrivLeak (privacy leakage detection)
- `knowmem_f`: KnowMem Forget (knowledge memorization on forget set)
- `knowmem_r`: KnowMem Retain (utility - knowledge retention on retain set)

We can evaluate models with different quantization settings:
- Full precision: `quantize_4bit=0, quantize_8bit=0`
- 4-bit quantization: `quantize_4bit=1, quantize_8bit=0`
- 8-bit quantization: `quantize_4bit=0, quantize_8bit=1`


In [5]:
!pip install pandas

  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 49.0 MB/s  0:00:00eta 0:00:01
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]


In [13]:
# Detailed Results Analysis and Insights
import pandas as pd
import numpy as np
OUTPUT_FILE = "output.csv"
output_file = repo_dir / OUTPUT_FILE

if output_file.exists():
    df = pd.read_csv(output_file)
    
    print("=" * 80)
    print("📊 DETAILED RESULTS ANALYSIS")
    print("=" * 80)
    print()
    
    # Define metric categories
    unlearning_metrics = ['verbmem_f', 'privleak', 'knowmem_f']  # Lower is better
    utility_metrics = ['knowmem_r', 'gen', 'tru', 'fac', 'flu']  # Higher is better
    
    for idx, row in df.iterrows():
        model_name = row['name']
        print(f"🔍 Model: {model_name}")
        print("-" * 80)
        
        # Unlearning Effectiveness (Lower is Better)
        print("\n✅ UNLEARNING EFFECTIVENESS (Lower is Better):")
        print("   Measures how well the model 'forgot' the target data")
        print()
        
        verbmem = row.get('verbmem_f', 0)
        privleak = row.get('privleak', 0)
        knowmem_f = row.get('knowmem_f', 0)
        
        print(f"   • verbmem_f (VerbMem Forget): {verbmem:.2f}%")
        if verbmem < 20:
            print("     ✓ Excellent: Model cannot reproduce verbatim text")
        elif verbmem < 40:
            print("     ⚠ Moderate: Some verbatim memory remains")
        else:
            print("     ✗ Poor: Model still memorizes verbatim text")
        
        print(f"   • privleak (Privacy Leakage): {privleak:.2f}%")
        if privleak < -50:
            print("     ✓ Excellent: Strong privacy protection")
        elif privleak < 0:
            print("     ✓ Good: Better than retrained baseline")
        elif privleak < 20:
            print("     ⚠ Moderate: Some privacy leakage")
        else:
            print("     ✗ Poor: Significant privacy leakage detected")
        
        print(f"   • knowmem_f (KnowMem Forget): {knowmem_f:.2f}%")
        if knowmem_f < 20:
            print("     ✓ Excellent: Model cannot answer questions about forget set")
        elif knowmem_f < 40:
            print("     ⚠ Moderate: Some semantic knowledge remains")
        else:
            print("     ✗ Poor: Model still has semantic knowledge")
        
        # Utility Preservation (Higher is Better)
        print("\n✅ UTILITY PRESERVATION (Higher is Better):")
        print("   Measures how well the model retained general capabilities")
        print()
        
        knowmem_r = row.get('knowmem_r', 0)
        gen = row.get('gen', 0)
        tru = row.get('tru', 0)
        fac = row.get('fac', 0)
        flu = row.get('flu', 0)
        
        print(f"   • knowmem_r (KnowMem Retain): {knowmem_r:.2f}%")
        if knowmem_r > 80:
            print("     ✓ Excellent: Utility well preserved")
        elif knowmem_r > 60:
            print("     ⚠ Moderate: Some utility loss")
        elif knowmem_r > 40:
            print("     ⚠ Significant: Notable utility degradation")
        else:
            print("     ✗ Poor: Catastrophic forgetting - model lost too much utility")
        
        if gen > 0:
            print(f"   • gen (MMLU - General Knowledge): {gen:.2f}")
            if gen > 0.6:
                print("     ✓ Good: Model retains general knowledge")
            else:
                print("     ⚠ Low: General knowledge degraded")
        
        if tru > 0:
            print(f"   • tru (TruthfulQA): {tru:.2f}")
            if tru > 0.6:
                print("     ✓ Good: Model remains truthful")
            else:
                print("     ⚠ Low: Truthfulness may be affected")
        
        if fac > 0:
            print(f"   • fac (TriviaQA - Factual): {fac:.2f}")
            if fac > 0.5:
                print("     ✓ Good: Factual knowledge retained")
            else:
                print("     ⚠ Low: Factual knowledge degraded")
        
        if flu > 0:
            print(f"   • flu (Fluency): {flu:.2f}")
            if flu > 0.7:
                print("     ✓ Good: Model remains fluent")
            else:
                print("     ⚠ Low: Fluency may be affected")
        
        # Overall Assessment
        print("\n📈 OVERALL ASSESSMENT:")
        print("-" * 80)
        
        # Calculate unlearning score (average of normalized unlearning metrics)
        unlearning_scores = []
        if verbmem < 100:
            unlearning_scores.append(1 - verbmem/100)  # Normalize to 0-1, higher is better
        if privleak < 0:
            unlearning_scores.append(1 - abs(privleak)/100)  # Negative privleak is good
        if knowmem_f < 100:
            unlearning_scores.append(1 - knowmem_f/100)
        
        avg_unlearning = np.mean(unlearning_scores) if unlearning_scores else 0
        
        # Calculate utility score
        utility_scores = []
        if knowmem_r > 0:
            utility_scores.append(knowmem_r/100)
        if gen > 0:
            utility_scores.append(gen)
        if tru > 0:
            utility_scores.append(tru)
        if fac > 0:
            utility_scores.append(fac)
        if flu > 0:
            utility_scores.append(flu)
        
        avg_utility = np.mean(utility_scores) if utility_scores else 0
        
        print(f"   Unlearning Effectiveness: {avg_unlearning*100:.1f}%")
        print(f"   Utility Preservation: {avg_utility*100:.1f}%")
        print()
        
        # Trade-off analysis
        if avg_unlearning > 0.7 and avg_utility > 0.7:
            print("   🎉 EXCELLENT: Strong unlearning with good utility preservation!")
        elif avg_unlearning > 0.7 and avg_utility < 0.5:
            print("   ⚠ WARNING: Good unlearning but catastrophic utility loss!")
            print("      This may indicate quantization issues or overly aggressive unlearning.")
        elif avg_unlearning < 0.5 and avg_utility > 0.7:
            print("   ⚠ WARNING: Good utility but poor unlearning!")
            print("      The model may not have forgotten the target data effectively.")
        elif avg_unlearning < 0.5 and avg_utility < 0.5:
            print("   ✗ POOR: Both unlearning and utility are low.")
            print("      The unlearning process may have failed or damaged the model.")
        else:
            print("   ⚠ MODERATE: Balanced but not optimal performance.")
        
        print()
        print("=" * 80)
        print()
    
    # Comparative Analysis (if multiple models)
    if len(df) > 1:
        print("📊 COMPARATIVE ANALYSIS")
        print("=" * 80)
        print()
        print("Comparing all models:")
        print()
        
        # Find best unlearning
        best_unlearning = df.loc[df['verbmem_f'].idxmin()] if 'verbmem_f' in df.columns else None
        if best_unlearning is not None:
            print(f"   Best Unlearning (lowest verbmem_f): {best_unlearning['name']} ({best_unlearning['verbmem_f']:.2f}%)")
        
        # Find best utility
        best_utility = df.loc[df['knowmem_r'].idxmax()] if 'knowmem_r' in df.columns else None
        if best_utility is not None:
            print(f"   Best Utility (highest knowmem_r): {best_utility['name']} ({best_utility['knowmem_r']:.2f}%)")
        
        # Find best balance
        if 'verbmem_f' in df.columns and 'knowmem_r' in df.columns:
            df['balance_score'] = (1 - df['verbmem_f']/100) * (df['knowmem_r']/100)
            best_balance = df.loc[df['balance_score'].idxmax()]
            print(f"   Best Balance (unlearning × utility): {best_balance['name']} (score: {best_balance['balance_score']:.3f})")
        
        print()
        print("💡 TIP: For detailed insights, see RESULTS_INSIGHTS.md in the repository root.")
        print()
    
else:
    print(f"⚠ Results file {OUTPUT_FILE} not found yet.")
    print("Please run the evaluation step first.")


📊 DETAILED RESULTS ANALYSIS

🔍 Model: ga
--------------------------------------------------------------------------------

✅ UNLEARNING EFFECTIVENESS (Lower is Better):
   Measures how well the model 'forgot' the target data

   • verbmem_f (VerbMem Forget): 0.00%
     ✓ Excellent: Model cannot reproduce verbatim text
   • privleak (Privacy Leakage): 30.20%
     ✗ Poor: Significant privacy leakage detected
   • knowmem_f (KnowMem Forget): 0.00%
     ✓ Excellent: Model cannot answer questions about forget set

✅ UTILITY PRESERVATION (Higher is Better):
   Measures how well the model retained general capabilities

   • knowmem_r (KnowMem Retain): 0.00%
     ✗ Poor: Catastrophic forgetting - model lost too much utility

📈 OVERALL ASSESSMENT:
--------------------------------------------------------------------------------
   Unlearning Effectiveness: 100.0%
   Utility Preservation: 0.0%

   ⚠ WARNING: Good unlearning but catastrophic utility loss!
      This may indicate quantization issue

## Step 8: Analyze Results & Insights

This section helps you interpret your evaluation results and understand what they mean for LLM unlearning effectiveness.


In [15]:
# Evaluation configuration

# Models to evaluate (can be local paths or HuggingFace model IDs)
# For this example, we'll evaluate the original target model
# In practice, you would evaluate your unlearned models from the ckpt/ directory
MODEL_DIRS = [
    # f"muse-bench/MUSE-{CORPUS.capitalize()}_target",  # Original target model
    # Add your unlearned model paths here, e.g.:
    f"./ckpt/news/finetuned_combined",
]

# Names for each model (should match length of MODEL_DIRS)
MODEL_NAMES = [
    # "original_target",
    # Add names for your models, e.g.:
    f"finetuned_combined",
]

# Evaluation settings
EVAL_CORPUS = "news"
OUTPUT_FILE = "output.csv"
TOKENIZER_DIR_EVAL = 'meta-llama/Llama-2-7b-hf'
METRICS = ['verbmem_f', 'privleak', 'knowmem_f', 'knowmem_r']  # All metrics
QUANTIZE_4BIT = 0  # Set to 1 for 4-bit quantization
QUANTIZE_8BIT = 0  # Set to 1 for 8-bit quantization
TEMP_DIR = "temp"
MAX_SAMPLES = None # Set to a number (e.g., 10) to test with a small dataset first
                     # None = use full dataset (recommended for final evaluation)

print("Evaluation Configuration:")
print("=" * 60)
print(f"Models to evaluate: {len(MODEL_DIRS)}")
for name, model_dir in zip(MODEL_NAMES, MODEL_DIRS):
    print(f"  - {name}: {model_dir}")
print(f"Corpus: {EVAL_CORPUS}")
print(f"Metrics: {', '.join(METRICS)}")
print(f"Quantization: 4-bit={QUANTIZE_4BIT}, 8-bit={QUANTIZE_8BIT}")
print(f"Max samples: {MAX_SAMPLES if MAX_SAMPLES else 'Full dataset'}")
print(f"Output file: {OUTPUT_FILE}")
print("=" * 60)


Evaluation Configuration:
Models to evaluate: 1
  - finetuned_combined: ./ckpt/news/finetuned_combined
Corpus: news
Metrics: verbmem_f, privleak, knowmem_f, knowmem_r
Quantization: 4-bit=0, 8-bit=0
Max samples: Full dataset
Output file: output.csv


In [35]:
!pip install rouge

  Using cached rouge-1.0.1-py3-none-any.whl.metadata (4.1 kB)
Using cached rouge-1.0.1-py3-none-any.whl (13 kB)


In [34]:
import sys
!{sys.executable} -m pip install rouge-score

In [19]:
!pip install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 57.8 MB/s  0:00:00m0:00:0100:01


In [23]:
import sys
!{sys.executable} -m pip install scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 38.9 MB/s  0:00:00 eta 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scikit-learn] [scikit-learn]


In [16]:
!pip install transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [27]:
import sys
!{sys.executable} -m pip install datasets

  Using cached pyarrow-22.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.2 kB)
  Using cached xxhash-3.6.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.18-py312-none-any.whl.metadata (7.5 kB)
  Using cached aiohttp-3.13.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.1 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (20 kB)
  Using cached multidict-6.7.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (5.3 kB)
  Using cached propcache-0.4.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (13 kB)
  Using cached yarl-1.22.0-cp312-cp312-manylinux2014_x86_64.ma

In [30]:
!pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 5.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [peft]1/2 [peft]


In [32]:
!pip install trl>=0.8.1

In [41]:
!pip install accelerate

In [42]:
pip install -i https://pypi.org/simple/ bitsandbytes

Looking in indexes: https://pypi.org/simple/
Note: you may need to restart the kernel to use updated packages.


In [73]:
!pip install -i https://pypi.org/simple/ bitsandbytes

Looking in indexes: https://pypi.org/simple/


In [ ]:
# Run evaluation
# Note: Evaluation can also take a significant amount of time
# Each metric requires running inference on the model

import subprocess
import sys
import os
from pathlib import Path

# Check if required variables are defined (from Cell 17)
required_vars = ['repo_dir', 'MODEL_DIRS', 'MODEL_NAMES', 'EVAL_CORPUS', 
                 'OUTPUT_FILE', 'TOKENIZER_DIR_EVAL', 'METRICS', 'TEMP_DIR',
                 'QUANTIZE_4BIT', 'QUANTIZE_8BIT', 'MAX_SAMPLES']
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    print("Error: Missing required variables. Please run Cell 17 (Evaluation Configuration) first.")
    print(f"Missing variables: {', '.join(missing_vars)}")
    raise NameError(f"Missing variables: {', '.join(missing_vars)}. Please run Cell 17 first.")


# Set CUDA devices safely
cuda_env = globals().get("CUDA_VISIBLE_DEVICES") or os.environ.get("CUDA_VISIBLE_DEVICES")
if cuda_env:
    os.environ["CUDA_VISIBLE_DEVICES"] = cuda_env
else:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)

# Define eval script and Python executable
eval_script = repo_dir / "eval.py"
python_executable = sys.executable

# Alpha parameter required by load_model function
ALPHA_EVAL = 5  # Default alpha value for evaluation

# --- Build the command ---
cmd = (
    [python_executable, str(eval_script)]
    + ["--model_dirs"] + MODEL_DIRS
    + ["--names"] + MODEL_NAMES
    + ["--corpus", EVAL_CORPUS,
       "--out_file", OUTPUT_FILE,
       "--tokenizer_dir", TOKENIZER_DIR_EVAL,
       "--metrics"] + METRICS
    + ["--temp_dir", TEMP_DIR,
       "--quantize_4bit", str(int(QUANTIZE_4BIT)),
       "--quantize_8bit", str(int(QUANTIZE_8BIT)),
       "--alpha", str(ALPHA_EVAL)]
)

# Add max_samples if specified (for quick testing)
if MAX_SAMPLES is not None:
    cmd = cmd + ["--max_samples", str(MAX_SAMPLES)]

# --- Run the evaluation ---
print("Starting evaluation...")
print(f"Command: {' '.join(cmd)}\n")
if MAX_SAMPLES is not None:
    print(f"⚠ Quick test mode: Using only {MAX_SAMPLES} samples per metric")
    print("   This is much faster for testing. Set MAX_SAMPLES=None for full evaluation.\n")
else:

    print("⚠ This may take a long time depending on:")
    print("   - Number of models to evaluate")
    print("   - Model size")
    print("   - Number of metrics")
    print("   - GPU availability\n")

result = subprocess.run(cmd, cwd=str(repo_dir), capture_output=True, text=True)

if result.returncode == 0:
    print("✓ Evaluation completed successfully!")
    print(f"Results saved to: {OUTPUT_FILE}")
    if result.stdout:
        print("\nOutput:")
        print(result.stdout)
else:
    print("✗ Error during evaluation:")
    print("=" * 60)
    if result.stdout:
        print("STDOUT:")
        print(result.stdout)
    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)
    print("=" * 60)

    # --- Common error hints ---
    error_output = (result.stdout + result.stderr).lower()
    if "nameerror" in error_output or "name 'args'" in error_output:
        print("\n⚠ Hint: Check eval.py — replace any 'args' references with function parameters.")
    elif "import" in error_output or "module" in error_output:
        print("\n⚠ Hint: Import error detected. Ensure dependencies are installed (pip install -r requirements.txt).")
    elif "cuda" in error_output or "gpu" in error_output:

        
        
        print("\n⚠ Hint: GPU/CUDA issue detected. On macOS, set quantize_4bit=0 and quantize_8bit=0.")


Starting evaluation...
Command: /usr/local/bin/python /workspace/CS534L_Project/FailureLLMUnlearning/eval.py --model_dirs ./ckpt/news/finetuned_combined --names finetuned_combined --corpus news --out_file output.csv --tokenizer_dir meta-llama/Llama-2-7b-hf --metrics verbmem_f privleak knowmem_f knowmem_r --temp_dir temp --quantize_4bit 0 --quantize_8bit 0 --alpha 5

⚠ This may take a long time depending on:
   - Number of models to evaluate
   - Model size
   - Number of metrics
   - GPU availability



In [74]:
!pip install accelerate

In [75]:
import sys
!{sys.executable} -m pip install --upgrade accelerate
!{sys.executable} -m pip install -i https://pypi.org/simple/ bitsandbytes

  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.12.0-py3-none-any.whl (380 kB)
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.29.0
    Uninstalling accelerate-0.29.0:
      Successfully uninstalled accelerate-0.29.0
Looking in indexes: https://pypi.org/simple/


In [21]:
# Run evaluation
# Note: Evaluation can also take a significant amount of time
# Each metric requires running inference on the model

import subprocess
import sys
import os
from pathlib import Path

# Check if required variables are defined (from Cell 17)
required_vars = ['repo_dir', 'MODEL_DIRS', 'MODEL_NAMES', 'EVAL_CORPUS', 
                 'OUTPUT_FILE', 'TOKENIZER_DIR_EVAL', 'METRICS', 'TEMP_DIR',
                 'QUANTIZE_4BIT', 'QUANTIZE_8BIT', 'MAX_SAMPLES']
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    print("Error: Missing required variables. Please run Cell 17 (Evaluation Configuration) first.")
    print(f"Missing variables: {', '.join(missing_vars)}")
    raise NameError(f"Missing variables: {', '.join(missing_vars)}. Please run Cell 17 first.")

# Set CUDA devices safely
CUDA_VISIBLE_DEVICES = os.environ.get("CUDA_VISIBLE_DEVICES", "")
if 'CUDA_VISIBLE_DEVICES' in globals() and globals()['CUDA_VISIBLE_DEVICES']:
    CUDA_VISIBLE_DEVICES = globals()['CUDA_VISIBLE_DEVICES']

# Apply CUDA device visibility
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES

# Define eval script and Python executable
eval_script = repo_dir / "eval.py"
python_executable = sys.executable

# Alpha parameter required by load_model function
ALPHA_EVAL = 5  # Default alpha value for evaluation

# --- Build the command ---
cmd = (
    [python_executable, str(eval_script)]
    + ["--model_dirs"] + MODEL_DIRS
    + ["--names"] + MODEL_NAMES
    + ["--corpus", EVAL_CORPUS,
       "--out_file", OUTPUT_FILE,
       "--tokenizer_dir", TOKENIZER_DIR_EVAL,
       "--metrics"] + METRICS
    + ["--temp_dir", TEMP_DIR,
       "--quantize_4bit", str(int(QUANTIZE_4BIT)),
       "--quantize_8bit", str(int(QUANTIZE_8BIT)),
       "--alpha", str(ALPHA_EVAL)]
)

# Add max_samples if specified (for quick testing)
if MAX_SAMPLES is not None:
    cmd = cmd + ["--max_samples", str(MAX_SAMPLES)]

# --- Run the evaluation ---
print("Starting evaluation...")
print(f"Command: {' '.join(cmd)}\n")
if MAX_SAMPLES is not None:
    print(f"⚠ Quick test mode: Using only {MAX_SAMPLES} samples per metric")
    print("   This is much faster for testing. Set MAX_SAMPLES=None for full evaluation.\n")
else:
    print("⚠ This may take a long time depending on:")
    print("   - Number of models to evaluate")
    print("   - Model size")
    print("   - Number of metrics")
    print("   - GPU availability\n")

result = subprocess.run(cmd, cwd=str(repo_dir), capture_output=True, text=True)

if result.returncode == 0:
    print("✓ Evaluation completed successfully!")
    print(f"Results saved to: {OUTPUT_FILE}")
    if result.stdout:
        print("\nOutput:")
        print(result.stdout)
else:
    print("✗ Error during evaluation:")
    print("=" * 60)
    if result.stdout:
        print("STDOUT:")
        print(result.stdout)
    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)
    print("=" * 60)

    # --- Common error hints ---
    error_output = (result.stdout + result.stderr).lower()
    if "nameerror" in error_output or "name 'args'" in error_output:
        print("\n⚠ Hint: Check eval.py — replace any 'args' references with function parameters.")
    elif "import" in error_output or "module" in error_output:
        print("\n⚠ Hint: Import error detected. Ensure dependencies are installed (pip install -r requirements.txt).")
    elif "cuda" in error_output or "gpu" in error_output:
        print("\n⚠ Hint: GPU/CUDA issue detected. On macOS, set quantize_4bit=0 and quantize_8bit=0.")


Starting evaluation...
Command: /usr/local/bin/python /workspace/CS534L_Project/FailureLLMUnlearning/eval.py --model_dirs ./ckpt/news/ga --names ga --corpus news --out_file output.csv --tokenizer_dir meta-llama/Llama-2-7b-hf --metrics verbmem_f privleak knowmem_f knowmem_r --temp_dir temp --quantize_4bit 1 --quantize_8bit 0 --alpha 5

⚠ This may take a long time depending on:
   - Number of models to evaluate
   - Model size
   - Number of metrics
   - GPU availability



KeyboardInterrupt: 

## Step 6b: Evaluate GA + GDR + Hinge Loss (ga_gdr_q4) Model

Evaluate the unlearned model trained with GA + GDR + Hinge Loss algorithm.


In [ ]:
# Evaluation configuration for GA + GDR + Hinge Loss (ga_gdr_q4)

# Models to evaluate - include the ga_gdr_q4 model
MODEL_DIRS_Q4 = [
    f"./ckpt/{CORPUS_Q4}/{ALGO_Q4}",
    # Optionally compare with baseline:
    # f"muse-bench/MUSE-{CORPUS_Q4.capitalize()}_target",  # Original target model
    # f"./ckpt/{CORPUS_Q4}/ga_gdr",  # GA+GDR without hinge loss for comparison
]

# Names for each model (should match length of MODEL_DIRS_Q4)
MODEL_NAMES_Q4 = [
    ALGO_Q4,
    # "original_target",
    # "ga_gdr",
]

# Evaluation settings
EVAL_CORPUS_Q4 = CORPUS_Q4
OUTPUT_FILE_Q4 = f"output_{ALGO_Q4}.csv"
TOKENIZER_DIR_EVAL_Q4 = 'meta-llama/Llama-2-7b-hf'
METRICS_Q4 = ['verbmem_f', 'privleak', 'knowmem_f', 'knowmem_r']  # All metrics
QUANTIZE_4BIT_Q4 = 0  # Set to 1 for 4-bit quantization evaluation
QUANTIZE_8BIT_Q4 = 0  # Set to 1 for 8-bit quantization evaluation
TEMP_DIR_Q4 = f"temp_{ALGO_Q4}"
MAX_SAMPLES_Q4 = None  # Set to a number (e.g., 10) to test with a small dataset first
                     # None = use full dataset (recommended for final evaluation)

print("GA + GDR + Hinge Loss (ga_gdr_q4) Evaluation Configuration:")
print("=" * 60)
print(f"Models to evaluate: {len(MODEL_DIRS_Q4)}")
for name, model_dir in zip(MODEL_NAMES_Q4, MODEL_DIRS_Q4):
    print(f"  - {name}: {model_dir}")
print(f"Corpus: {EVAL_CORPUS_Q4}")
print(f"Metrics: {', '.join(METRICS_Q4)}")
print(f"Quantization: 4-bit={QUANTIZE_4BIT_Q4}, 8-bit={QUANTIZE_8BIT_Q4}")
print(f"Max samples: {MAX_SAMPLES_Q4 if MAX_SAMPLES_Q4 else 'Full dataset'}")
print(f"Output file: {OUTPUT_FILE_Q4}")
print("=" * 60)


In [ ]:
# Run evaluation for GA + GDR + Hinge Loss (ga_gdr_q4) model
# Note: Evaluation can also take a significant amount of time
# Each metric requires running inference on the model

import subprocess
import sys
import os
from pathlib import Path

# Check if required variables are defined
required_vars_q4 = ['repo_dir', 'MODEL_DIRS_Q4', 'MODEL_NAMES_Q4', 'EVAL_CORPUS_Q4', 
                    'OUTPUT_FILE_Q4', 'TOKENIZER_DIR_EVAL_Q4', 'METRICS_Q4', 'TEMP_DIR_Q4',
                    'QUANTIZE_4BIT_Q4', 'QUANTIZE_8BIT_Q4', 'MAX_SAMPLES_Q4']
missing_vars_q4 = [v for v in required_vars_q4 if v not in globals()]
if missing_vars_q4:
    print("Error: Missing required variables. Please run the evaluation configuration cell first.")
    print(f"Missing variables: {', '.join(missing_vars_q4)}")
    raise NameError(f"Missing variables: {', '.join(missing_vars_q4)}. Please run the evaluation configuration cell first.")

# Set CUDA devices safely
cuda_env_q4 = globals().get("CUDA_VISIBLE_DEVICES_Q4") or os.environ.get("CUDA_VISIBLE_DEVICES")
if cuda_env_q4:
    os.environ["CUDA_VISIBLE_DEVICES"] = cuda_env_q4
else:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)

# Define eval script and Python executable
eval_script = repo_dir / "eval.py"
python_executable = sys.executable

# Alpha parameter required by load_model function
ALPHA_EVAL_Q4 = 5  # Default alpha value for evaluation

# --- Build the command ---
cmd_q4_eval = (
    [python_executable, str(eval_script)]
    + ["--model_dirs"] + MODEL_DIRS_Q4
    + ["--names"] + MODEL_NAMES_Q4
    + ["--corpus", EVAL_CORPUS_Q4,
       "--out_file", OUTPUT_FILE_Q4,
       "--tokenizer_dir", TOKENIZER_DIR_EVAL_Q4,
       "--metrics"] + METRICS_Q4
    + ["--temp_dir", TEMP_DIR_Q4,
       "--quantize_4bit", str(int(QUANTIZE_4BIT_Q4)),
       "--quantize_8bit", str(int(QUANTIZE_8BIT_Q4)),
       "--alpha", str(ALPHA_EVAL_Q4)]
)

# Add max_samples if specified (for quick testing)
if MAX_SAMPLES_Q4 is not None:
    cmd_q4_eval = cmd_q4_eval + ["--max_samples", str(MAX_SAMPLES_Q4)]

# --- Run the evaluation ---
print("Starting evaluation for GA + GDR + Hinge Loss (ga_gdr_q4)...")
print(f"Command: {' '.join(cmd_q4_eval)}\n")
if MAX_SAMPLES_Q4 is not None:
    print(f"⚠ Quick test mode: Using only {MAX_SAMPLES_Q4} samples per metric")
    print("   This is much faster for testing. Set MAX_SAMPLES_Q4=None for full evaluation.\n")
else:
    print("⚠ This may take a long time depending on:")
    print("   - Number of models to evaluate")
    print("   - Model size")
    print("   - Number of metrics")
    print("   - GPU availability\n")

result_q4_eval = subprocess.run(cmd_q4_eval, cwd=str(repo_dir), capture_output=True, text=True)

if result_q4_eval.returncode == 0:
    print("✓ Evaluation completed successfully!")
    print(f"Results saved to: {OUTPUT_FILE_Q4}")
    if result_q4_eval.stdout:
        print("\nOutput:")
        print(result_q4_eval.stdout)
else:
    print("✗ Error during evaluation:")
    print("=" * 60)
    if result_q4_eval.stdout:
        print("STDOUT:")
        print(result_q4_eval.stdout)
    if result_q4_eval.stderr:
        print("\nSTDERR:")
        print(result_q4_eval.stderr)
    print("=" * 60)

    # --- Common error hints ---
    error_output_q4 = (result_q4_eval.stdout + result_q4_eval.stderr).lower()
    if "nameerror" in error_output_q4 or "name 'args'" in error_output_q4:
        print("\n⚠ Hint: Check eval.py — replace any 'args' references with function parameters.")
    elif "import" in error_output_q4 or "module" in error_output_q4:
        print("\n⚠ Hint: Import error detected. Ensure dependencies are installed (pip install -r requirements.txt).")
    elif "cuda" in error_output_q4 or "gpu" in error_output_q4:
        print("\n⚠ Hint: GPU/CUDA issue detected. On macOS, set quantize_4bit=0 and quantize_8bit=0.")


## Step 7: View Results

After evaluation completes, we can load and display the results from the output CSV file.


In [22]:
# Load and display results
import pandas as pd
import os

output_file = repo_dir / OUTPUT_FILE

if output_file.exists():
    print(f"Loading results from {OUTPUT_FILE}...\n")
    df = pd.read_csv(output_file)
    
    # Display the results
    print("Evaluation Results:")
    print("=" * 80)
    print(df.to_string(index=False))
    print("=" * 80)
    
    # Display summary statistics if multiple models
    if len(df) > 1:
        print("\nSummary Statistics:")
        print("=" * 80)
        numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
        if len(numeric_cols) > 0:
            print(df[numeric_cols].describe())
    
else:
    print(f"⚠ Results file {OUTPUT_FILE} not found yet.")
    print("Please run the evaluation step first.")


Loading results from output.csv...

Evaluation Results:
name  verbmem_f  privleak  knowmem_f  knowmem_r  gen  tru  fac  flu
  ga        0.0 30.196982        0.0        0.0  0.0  0.0  0.0  0.0


## Additional Notes and Tips

### Running Multiple Algorithms

To run multiple unlearning algorithms, you can modify the configuration in Step 5:

```python
algorithms = ['ga', 'ga_gdr', 'npo', 'npo_gdr']
for algo in algorithms:
    # Run unlearning for each algorithm
    ...
```

### Using Different Corpora

You can switch between News and Books corpora by changing:
```python
CORPUS = 'books'  # or 'news'
```

### Quantization Testing

To test models with different quantization settings, modify the evaluation step:
- Full precision: `QUANTIZE_4BIT=0, QUANTIZE_8BIT=0`
- 4-bit: `QUANTIZE_4BIT=1, QUANTIZE_8BIT=0`
- 8-bit: `QUANTIZE_4BIT=0, QUANTIZE_8BIT=1`

### GPU Requirements

- **Recommended**: NVIDIA GPU with 16GB+ VRAM
- For smaller GPUs, reduce `PER_DEVICE_BATCH_SIZE` or use quantization
- For CPU-only, expect significantly longer training times

### Time Estimates

- **Data loading**: 5-15 minutes
- **Unlearning (1 epoch)**: 1-4 hours (depending on GPU)
- **Evaluation**: 30 minutes - 2 hours per model

### Troubleshooting

1. **HuggingFace access**: Make sure you're logged in: `huggingface-cli login`
2. **CUDA errors**: Check GPU availability and CUDA installation
3. **Memory errors**: Reduce batch size or use gradient checkpointing
4. **Import errors**: Ensure all dependencies are installed correctly

### Next Steps

1. Experiment with different algorithms
2. Try different hyperparameters (learning rate, epochs, alpha, threshold)
3. Compare results across different quantization settings
4. Analyze the trade-offs between unlearning effectiveness and model utility


## Quick Reference: Shell Commands

If you prefer to run commands directly in the terminal instead of through this notebook, here are the key commands:

### 1. Clone Repository
```bash
cd /Users/himanshumishra/Library/CloudStorage/OneDrive-UBC/UBC/Term1/Projects
git clone https://github.com/zzwjames/FailureLLMUnlearning.git
cd FailureLLMUnlearning
```

### 2. Create Conda Environment
```bash
conda env create -f environment.yml
conda activate py310
```

### 3. Install Dependencies (Alternative to conda)
```bash
pip install -r requirements.txt
```

### 4. Load Data
```bash
python load_data.py
```

### 5. Run Unlearning (Example)
```bash
cd baselines
python unlearn.py \
    --algo ga \
    --model_dir muse-bench/MUSE-News_target \
    --tokenizer_dir meta-llama/Llama-2-7b-hf \
    --data_file ../data/news/raw/forget.txt \
    --retain_data_file ../data/news/raw/retain1.txt \
    --out_dir ../ckpt/news/ga \
    --max_len 2048 \
    --epochs 10 \
    --lr 1e-5 \
    --alpha 1 \
    --threshold 90 \
    --per_device_batch_size 2
```

### 6. Evaluate Models
```bash
cd ..  # Back to repository root
python eval.py \
    --model_dirs "muse-bench/MUSE-News_target" \
    --names "original" \
    --corpus news \
    --out_file "output.csv" \
    --quantize_4bit 0 \
    --quantize_8bit 0
```


## Step 9: Upload Unlearned Model to Hugging Face

This step shows how to upload a trained/unlearned model checkpoint from `ckpt/news/ga` to your Hugging Face Hub account.

Run the next cell after you have:
- Created a Hugging Face account
- Logged in from this environment using `huggingface-cli login` or `login()` from `huggingface_hub`
- Finished training an unlearned model in `ckpt/news/ga`.



In [ ]:
!pip install huggingface_hub

  Using cached hf_xet-1.2.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
Using cached hf_xet-1.2.0-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
Using cached click-8.3.1-py3-none-any.whl (108 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [huggingface_hub] [huggingface_hub]


In [ ]:
     from huggingface_hub import login
     login(token="token write")

In [ ]:
!pip install torch

In [ ]:
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |

In [11]:
# Upload unlearned model from `ckpt/news/ga` to Hugging Face Hub

from pathlib import Path
from huggingface_hub import HfApi, create_repo, upload_folder, whoami

# 1. Set local model path and target repo name
#    - `local_dir`: folder that contains your final unlearned checkpoint (adjust if needed)
#    - `repo_id`: "username/repo_name" on Hugging Face (replace with YOUR values)

local_dir = Path("/workspace/CS534L_Project/FailureLLMUnlearning/ckpt")
repo_id = "himishra/CS532L"  # TODO: change this

if not local_dir.exists():
    raise FileNotFoundError(f"Local model directory not found: {local_dir}")

# 2. Verify you are logged in
try:
    info = whoami()
    print(f"Authenticated to Hugging Face Hub as: {info.get('name', info.get('username'))}")
except Exception as e:
    raise RuntimeError(
        "You are not logged in to Hugging Face Hub.\n"
        "Run `huggingface-cli login` in a terminal, or use `from huggingface_hub import login; login(token=...)`."
    ) from e

# 3. Create the repo if it does not exist yet
api = HfApi()

print(f"Ensuring repo exists: {repo_id}")
create_repo(repo_id=repo_id, exist_ok=True)

# 4. Upload the entire folder (all files under ckpt/news/ga)
print(f"Uploading folder `{local_dir}` to `{repo_id}` (this may take a while)...")
upload_folder(
    repo_id=repo_id,
    folder_path=str(local_dir),
    commit_message="Upload unlearned model checkpoint from FailureLLMUnlearning",
)

print("\n✓ Upload complete! Your model is now available at:")
print(f"  https://huggingface.co/{repo_id}")


Authenticated to Hugging Face Hub as: himishra
Ensuring repo exists: himishra/CS532L
Uploading folder `/workspace/CS534L_Project/FailureLLMUnlearning/ckpt` to `himishra/CS532L` (this may take a while)...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✓ Upload complete! Your model is now available at:
  https://huggingface.co/himishra/CS532L


In [ ]:
# Download model ckpt and data from Hugging Face to local directories

from pathlib import Path
from huggingface_hub import snapshot_download

# --- Configure what to download ---
# Replace these with your repos/paths on Hugging Face Hub
MODEL_REPO_ID = "your-username/your-model-repo"      # e.g., himishra/CS532L
DATA_REPO_ID = "your-username/your-data-repo"        # e.g., himishra/CS532L-data

# Local destinations
local_model_dir = Path(repo_dir) / "ckpt" / "hf_import/model"
local_data_dir = Path(repo_dir) / "data" / "hf_import"
local_model_dir.mkdir(parents=True, exist_ok=True)
local_data_dir.mkdir(parents=True, exist_ok=True)

print("Starting download from Hugging Face...")
print("Model repo:", MODEL_REPO_ID)
print("Data repo:", DATA_REPO_ID)
print("Model dest:", local_model_dir)
print("Data dest:", local_data_dir)

# Download model checkpoint (full snapshot)
try:
    snapshot_download(
        repo_id=MODEL_REPO_ID,
        local_dir=str(local_model_dir),
        ignore_patterns=["*.pt.index.json", "*.msgpack"],  # optional: skip huge index files
        resume_download=True,
    )
    print("✓ Model downloaded")
except Exception as e:
    print("✗ Failed to download model:", e)

# Download data snapshot
try:
    snapshot_download(
        repo_id=DATA_REPO_ID,
        local_dir=str(local_data_dir),
        resume_download=True,
    )
    print("✓ Data downloaded")
except Exception as e:
    print("✗ Failed to download data:", e)

print("Done. You can now load from:")
print("  Model:", local_model_dir)
print("  Data:", local_data_dir)


In [ ]:
# Download specific files/folders from a Hugging Face repo (no full clone)

from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

# --- Configure what to pull ---
REPO_ID = "your-username/your-repo"  # e.g., himishra/CS532L
# List the exact paths (relative to the repo) you want. Use forward slashes.
FILES_OR_PATTERNS = [
    "ckpt/model.safetensors",      # single file example
    "data/subfolder/*",            # folder or glob example (optional)
]
# Where to place them locally
DEST_DIR = Path(repo_dir) / "hf_selected"
DEST_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading selected files/patterns from:", REPO_ID)
print("Target directory:", DEST_DIR)

# Approach 1: Use snapshot_download with allow_patterns for globs/folders
try:
    snapshot_download(
        repo_id=REPO_ID,
        local_dir=str(DEST_DIR),
        allow_patterns=FILES_OR_PATTERNS,
        resume_download=True,
        local_dir_use_symlinks=False,  # store real files
    )
    print("✓ Downloaded requested files/patterns via snapshot_download")
except Exception as e:
    print("⚠ snapshot_download failed (will try hf_hub_download per file):", e)
    # Fallback: download file-by-file (works only for explicit file paths, not globs)
    for rel_path in FILES_OR_PATTERNS:
        if "*" in rel_path or rel_path.endswith("/"):
            print(f"  Skipping glob/folder '{rel_path}' in fallback mode; provide explicit file paths.")
            continue
        try:
            local_path = hf_hub_download(
                repo_id=REPO_ID,
                filename=rel_path,
                local_dir=str(DEST_DIR),
                local_dir_use_symlinks=False,
                resume_download=True,
            )
            print(f"  ✓ Downloaded {rel_path} -> {local_path}")
        except Exception as ex:
            print(f"  ✗ Failed to download {rel_path}: {ex}")

print("Done. Files are in:", DEST_DIR)
